# WLA / DAP Historical Curve Reconstruction

This notebook reconstructs the historical Bloomberg `WLA Index` futures chain used as the DAP / ID x IPCA real-rate curve input for the thesis.

**Thesis window:** January 2010 through June 2025, with **186 monthly curve observations**.

## Important date convention

The monthly observation is the **actual usable WLA/DAP market date for the month**, not mechanically the last weekday returned by a generic business-month-end calendar.

The initial Bloomberg extraction used 186 weekday business-month-end candidate dates. A subsequent audit found **21 candidate dates for which the WLA chain existed but every `PX_LAST` was missing**. Those months were re-queried on the preceding usable trading date and validated directly in Bloomberg/BQuant.

Therefore, the final historical panel still contains **186 monthly observations**, but 21 observation dates differ from the original generic month-end candidates.

A reviewer without Bloomberg can independently validate the underlying DAP market data using B3 historical **BVBG.086.01 PriceReport** files. See `README_WLA_DAP_REPLICATION.md`.


## Bloomberg Excel equivalent

For a single historical date, Bloomberg Excel can return the live futures chain with:

```excel
=BDS("WLA Index","FUT_CHAIN","CHAIN_DATE=20240131","INCLUDE_EXPIRED_CONTRACTS=N")
```

Change `CHAIN_DATE` to the historical date to be replicated.

The number of live contracts is **not constant over time**. Generic positions (`WL1`, `WL2`, ...) are rolling chain positions and must not be interpreted as fixed monthly maturities.


In [ ]:
import bql
import pandas as pd
from pathlib import Path

from download_file import download_file

bq = bql.Service()


## Fetch the historical WLA futures chain for one date

The function below:

1. requests the historical `WLA Index` futures universe;
2. retrieves the specific contract ticker, `PX_LAST`, and `FUTURES_VALUATION_DATE`;
3. sorts contracts by their actual valuation date;
4. assigns `CHAIN_POSITION` only after the true maturity ordering is known;
5. preserves missing, zero, negative, and positive `PX_LAST` observations.

No sign filter is applied at the acquisition stage.


In [ ]:
def fetch_wla_chain_full(d):
    """Return the historical WLA futures chain for one Bloomberg as-of date."""

    chain_date = pd.Timestamp(d).normalize()
    bql_date = chain_date.strftime("%Y-%m-%d")

    universe = bq.univ.futures(
        "WLA Index",
        dates=bql_date,
    )

    items = {
        "Ticker": bq.data.id(),
        "PX_LAST": bq.data.px_last(dates=bql_date),
        "FUTURES_VALUATION_DATE": bq.data.futures_valuation_date(),
    }

    resp = bq.execute(
        bql.Request(universe, items)
    )

    out = pd.concat(
        [x.df() for x in resp],
        axis=1,
    )

    required = [
        "Ticker",
        "PX_LAST",
        "FUTURES_VALUATION_DATE",
    ]

    missing_columns = [c for c in required if c not in out.columns]
    if missing_columns:
        raise ValueError(
            f"Bloomberg response is missing columns: {missing_columns}"
        )

    out = out[required].copy()

    out["Ticker"] = out["Ticker"].astype(str).str.strip()
    out["PX_LAST"] = pd.to_numeric(out["PX_LAST"], errors="coerce")
    out["FUTURES_VALUATION_DATE"] = pd.to_datetime(
        out["FUTURES_VALUATION_DATE"],
        errors="coerce",
    )

    # Contract order is the actual futures valuation/maturity order.
    out = (
        out
        .sort_values("FUTURES_VALUATION_DATE", na_position="last")
        .reset_index(drop=True)
    )

    out["CHAIN_POSITION"] = range(1, len(out) + 1)
    out["CHAIN_DATE"] = chain_date

    return out[
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "Ticker",
            "FUTURES_VALUATION_DATE",
            "PX_LAST",
        ]
    ]


## Step 1 — Generate the 186 candidate month-end dates

`BMonthEnd` provides the initial weekday business-month-end candidates. These are **candidate query dates**, not automatically the final market observation dates.

The audit below is what detects months in which the candidate date has no usable WLA settlement observations.


In [ ]:
candidate_dates = pd.date_range(
    start="2010-01-01",
    end="2025-06-30",
    freq=pd.offsets.BMonthEnd(),
)

print("Candidate dates:", len(candidate_dates))
print("First candidate:", candidate_dates[0])
print("Last candidate:", candidate_dates[-1])


## Step 2 — Query all candidate dates and audit quote availability

A date is considered unusable for the monthly curve if the historical chain is returned but **all contracts have `PX_LAST = NaN`**.

This distinction matters: individual contracts may legitimately have missing quotes while other contracts on the same date remain usable.


In [ ]:
candidate_frames = []
errors = []

for i, d in enumerate(candidate_dates, start=1):
    try:
        x = fetch_wla_chain_full(d)
        candidate_frames.append(x)

        print(
            f"[{i:03d}/{len(candidate_dates)}] "
            f"{d:%Y-%m-%d} -> "
            f"{len(x)} contracts, "
            f"{x['PX_LAST'].notna().sum()} usable PX_LAST"
        )

    except Exception as exc:
        errors.append({
            "CHAIN_DATE": d,
            "ERROR": str(exc),
        })
        print(
            f"[{i:03d}/{len(candidate_dates)}] "
            f"{d:%Y-%m-%d} -> ERROR: {exc}"
        )

candidate_raw = pd.concat(
    candidate_frames,
    ignore_index=True,
)

errors_df = pd.DataFrame(errors)

candidate_audit = (
    candidate_raw
    .groupby("CHAIN_DATE")
    .agg(
        rows=("Ticker", "size"),
        nonmissing_px=("PX_LAST", "count"),
    )
    .sort_index()
)

empty_candidate_dates = candidate_audit.index[
    candidate_audit["nonmissing_px"].eq(0)
]

print("\nCandidate dates requested:", len(candidate_dates))
print("Candidate dates returned:", candidate_raw["CHAIN_DATE"].nunique())
print("Query errors:", len(errors_df))
print("Dates with zero usable PX_LAST:", len(empty_candidate_dates))
print(pd.Series(empty_candidate_dates).to_string(index=False))


## Step 3 — Validated date replacements

The audit identified **21 nominal month-end candidate dates** with no usable WLA `PX_LAST`. Each was re-queried on the preceding usable WLA/DAP trading date.

The mapping is kept explicitly in the notebook as an audit trail. It should not be inferred solely from an ANBIMA holiday calendar because B3/DAP trading closures — especially year-end closures — are not always represented by the same calendar convention used for BUS/252 day-count calculations.


In [ ]:
date_replacements = pd.DataFrame(
    [
        ("2010-12-31", "2010-12-30"),
        ("2011-12-30", "2011-12-29"),
        ("2012-12-31", "2012-12-28"),
        ("2013-03-29", "2013-03-28"),
        ("2013-12-31", "2013-12-30"),
        ("2014-12-31", "2014-12-30"),
        ("2015-12-31", "2015-12-30"),
        ("2016-12-30", "2016-12-29"),
        ("2017-02-28", "2017-02-24"),
        ("2017-12-29", "2017-12-28"),
        ("2018-03-30", "2018-03-29"),
        ("2018-05-31", "2018-05-30"),
        ("2018-12-31", "2018-12-28"),
        ("2019-12-31", "2019-12-30"),
        ("2020-12-31", "2020-12-30"),
        ("2021-12-31", "2021-12-30"),
        ("2022-02-28", "2022-02-25"),
        ("2022-12-30", "2022-12-29"),
        ("2023-12-29", "2023-12-28"),
        ("2024-03-29", "2024-03-28"),
        ("2024-12-31", "2024-12-30"),
    ],
    columns=["ORIGINAL_CANDIDATE_DATE", "REPLACEMENT_TRADING_DATE"],
)

for col in date_replacements.columns:
    date_replacements[col] = pd.to_datetime(date_replacements[col])

print(date_replacements.to_string(index=False))

# Audit: the hard-coded mapping should correspond exactly to the
# all-missing candidate dates discovered above.
expected_empty = set(date_replacements["ORIGINAL_CANDIDATE_DATE"])
observed_empty = set(empty_candidate_dates)

print("\nUnexpected empty candidate dates:", sorted(observed_empty - expected_empty))
print("Mapped dates not currently empty:", sorted(expected_empty - observed_empty))


## Step 4 — Re-query the 21 corrected trading dates

This is the acquisition code used to create `wla_replacement_dates_21_raw.xlsx`.

Every replacement date must return at least one non-missing `PX_LAST` before it is allowed into the final historical panel.


In [ ]:
replacement_frames = []

replacement_dates = date_replacements[
    "REPLACEMENT_TRADING_DATE"
].tolist()

for i, d in enumerate(replacement_dates, start=1):
    print(f"{i:02d}/{len(replacement_dates)}  {d:%Y-%m-%d}")
    replacement_frames.append(
        fetch_wla_chain_full(d)
    )

replacement_raw = pd.concat(
    replacement_frames,
    ignore_index=True,
)

replacement_audit = (
    replacement_raw
    .groupby("CHAIN_DATE")
    .agg(
        rows=("Ticker", "size"),
        nonmissing_px=("PX_LAST", "count"),
        min_px=("PX_LAST", "min"),
        max_px=("PX_LAST", "max"),
    )
    .sort_index()
)

print(replacement_audit.to_string())

if (replacement_audit["nonmissing_px"] <= 0).any():
    bad = replacement_audit.index[
        replacement_audit["nonmissing_px"] <= 0
    ]
    raise ValueError(
        f"Replacement dates still lacking usable PX_LAST: {list(bad)}"
    )


## Save the 21-date replacement audit workbook

This file is retained separately so the date correction is independently auditable.


In [ ]:
replacement_file = Path("wla_replacement_dates_21_raw.xlsx")

with pd.ExcelWriter(replacement_file) as writer:
    replacement_raw.to_excel(
        writer,
        sheet_name="bqnt output",
        index=False,
    )

# Creates a browser download button in the BQuant/Jupyter environment.
download_file(replacement_file)

print("Rows:", len(replacement_raw))
print("Dates:", replacement_raw["CHAIN_DATE"].nunique())


## Step 5 — Build the corrected 186-date raw historical panel

The 21 all-missing candidate-date blocks are removed and replaced by the newly queried trading-date blocks.

The replacement date itself remains the observation date. For example, the 2010 December observation is stored as `2010-12-30`, **not relabeled as `2010-12-31`**.


In [ ]:
obsolete_dates = set(
    date_replacements["ORIGINAL_CANDIDATE_DATE"]
)

candidate_kept = candidate_raw.loc[
    ~candidate_raw["CHAIN_DATE"].isin(obsolete_dates)
].copy()

full_chain = pd.concat(
    [candidate_kept, replacement_raw],
    ignore_index=True,
)

full_chain = (
    full_chain
    .sort_values(
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "FUTURES_VALUATION_DATE",
        ]
    )
    .reset_index(drop=True)
)

final_audit = (
    full_chain
    .groupby("CHAIN_DATE")
    .agg(
        rows=("Ticker", "size"),
        nonmissing_px=("PX_LAST", "count"),
    )
    .sort_index()
)

print("Final curve dates:", full_chain["CHAIN_DATE"].nunique())
print("Final contract rows:", len(full_chain))
print("Dates with zero usable PX_LAST:", int(final_audit["nonmissing_px"].eq(0).sum()))
print("Duplicate date/position pairs:", full_chain.duplicated(["CHAIN_DATE", "CHAIN_POSITION"]).sum())

assert full_chain["CHAIN_DATE"].nunique() == 186
assert final_audit["nonmissing_px"].gt(0).all()
assert not full_chain.duplicated(["CHAIN_DATE", "CHAIN_POSITION"]).any()


## Step 6 — Audit the corrected raw quotes

The raw panel must preserve legitimate positive, zero, and negative real rates. Only `NaN` denotes an unavailable individual quote.

For the corrected thesis-window snapshot used in the jury revision, the final acquisition contains:

- **186 usable monthly dates**;
- **3,144 contract-date rows**;
- **2,463 non-missing `PX_LAST` observations**;
- **681 missing individual `PX_LAST` observations**;
- **2 zero rates**;
- **99 negative rates**;
- **2,362 positive rates**.

These counts are audit statistics for the frozen thesis dataset, not filters to impose on future downloads.


In [ ]:
px = pd.to_numeric(
    full_chain["PX_LAST"],
    errors="coerce",
)

print("Total rows:", len(full_chain))
print("Usable PX_LAST:", px.notna().sum())
print("PX_LAST missing:", px.isna().sum())
print("PX_LAST equal to zero:", (px == 0).sum())
print("PX_LAST negative:", (px < 0).sum())
print("PX_LAST positive:", (px > 0).sum())

print("\nZero WLA rates:")
print(
    full_chain.loc[
        px.eq(0),
        [
            "CHAIN_DATE",
            "CHAIN_POSITION",
            "Ticker",
            "PX_LAST",
            "FUTURES_VALUATION_DATE",
        ],
    ].to_string(index=False)
)


## Step 7 — Export the authoritative corrected raw dataset

No tenor interpolation, rate-unit conversion, or missing-value imputation is performed here.

Those transformations belong in the thesis Python pipeline. In particular:

- `FUTURES_VALUATION_DATE` is the actual contract maturity/valuation date used to construct tenor;
- BUS/252 tenor is calculated downstream;
- Bloomberg/B3 percentage-point quotes are converted to decimal rates downstream;
- zero and negative rates are retained;
- individual `NaN` quotes are excluded only when constructing the usable surface.

The BQuant acquisition does not require volume for the IPCA curve. The production `load_ipca_surface()` routine does not use volume.


In [ ]:
excel_file = Path("wla_historical_chain_2010_2025_raw.xlsx")
csv_file = Path("wla_historical_chain_2010_2025_raw.csv")

with pd.ExcelWriter(excel_file) as writer:
    full_chain.to_excel(
        writer,
        sheet_name="bqnt output",
        index=False,
    )

full_chain.to_csv(
    csv_file,
    index=False,
)

download_file(excel_file)
download_file(csv_file)

print("Export completed.")
print("Excel:", excel_file)
print("CSV:", csv_file)
print("Rows:", len(full_chain))
print("Dates:", full_chain["CHAIN_DATE"].nunique())


## Downstream production workbook

The acquisition files above are the raw audit trail. The thesis production pipeline uses the legacy-compatible workbook:

`hist_ipca_curve_contracts_db.xlsx`

For the corrected jury-revision version, its 21 all-missing date blocks were replaced by the validated Bloomberg observations described above. The production workbook audit returns:

- **186 raw curve dates**;
- **186 usable curve dates** after `load_ipca_surface()`;
- **2,463 usable WLA observations**;
- no completely missing monthly curve date.

The production loader converts `PX_LAST` from percentage points to decimal rates and retains valid zero and negative real rates.


## B3 public validation note

A reviewer without Bloomberg can use B3's historical **Boletim de Negociação — BVBG.086.01 PriceReport** for the corresponding date and identify the DAP contract and official adjustment/settlement rate.

Example already cross-checked:

- Date: `2018-09-28`
- Bloomberg contract: `WLV18 Index`
- Bloomberg `PX_LAST`: `0.000`
- B3 DAP V18 official adjustment rate: `0.000`

Therefore, zero observations must not automatically be classified as missing.
